In [2]:
import numpy as np
from scipy.optimize import fsolve, root

import xarray as xr
import pandas as pd

### Importing the input dataset

In [15]:
# inputs from BoxToReal.ipynb
inputs = xr.open_dataset("clean_input_dataset.nc")
index_isf = inputs.Nisf.values
name_isf = inputs.name_isf.values

In [16]:
# Define reference properties and parameters 

#EOS
T_star = 0 # °C  (PICO)
S_star = 34 # PSU   (PICO)
rho_star = 1033 # kg/m^3   (PICO)
alpha = 7.5e-5 # /°C   (PICO)
beta = 7.7e-4 # /PSU (PICO)

def EOS(T,S):
    return rho_star*(1-alpha*(T-T_star)+beta*(S-S_star))
    
#Liquidus
la = -0.0572 # °C/PSU   (PICO)
lb = 0.0788 # °C   (PICO)
lc = 7.59e-4 # °C/m   (PICO->Burgard)

def liquidus(S,h):
    return la*S+lb-lc*h

#Water properties
L = 3.34e5 # J/kg   (PICO)
c_star = 3974 # J/kg/°C   (PICO)
lambd = L/c_star # 84 K

#Ice properties
rho_i = 910 # kg/m^3   (PICO)
nu = rho_i/rho_star #0.88

#PICO parameters
# Reese 2018
C = 1e6 #m^6/s/kg
gammaT = 2e-5 #m/s

#Vertical mixing
# ~Olbers & Hellmer 2010
kappa_diff = 1e-7  # m/s
kappa_conv = 1e-3 # m/s

#Polynia
g = 20

#AABW
C2 = 4e6 #m^6/s/kg

### PICO solver

In [17]:
# PICO solver from Reese 2019
def PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth):
    g1 = Ac*frac*gammaT # list of g1
    g2 = g1/nu/lambd
    s = Sd/nu/lambd
    # management of the 1st box
    T_st_0 = la*Sd+lb-lc*depth[0]-Td
    x0 = -g1[0]/(2*C*rho_star*(beta*s-alpha))+np.sqrt((g1[0]/(2*C*rho_star*(beta*s-alpha)))**2-g1[0]*T_st_0/(C*rho_star*(beta*s-alpha)))
    y0 = Sd*x0/nu/lambd
    T0 = Td-x0
    S0 = Sd-y0
    q = C*rho_star*(beta*s-alpha)*x0

    # management of the other boxes
    T = np.zeros(nbox)
    S = np.zeros(nbox)
    m = np.zeros(nbox)
    T[0] = T0
    S[0] = S0
    m[0] = -gammaT/nu/lambd*(la*S[0]+lb-lc*depth[0]-T[0])*3600*24*30 #m/30d
    for k in range(1,nbox):
        T_st = la*S[k-1]+lb-lc*depth[k]-T[k-1]
        x = -g1[k]*T_st/(q+g1[k]-g2[k]*la*S[k-1])
        y = S[k-1]*x/nu/lambd
        T[k] = T[k-1]-x
        S[k] = S[k-1]-y
        m[k] = -gammaT/nu/lambd*(la*S[k]+lb-lc*depth[k]-T[k])*3600*24*30 #m/30d

    
    
    return T,S,m,q

### Box model solver

In [18]:
# compute AABW flux [m^3/s]
def compute_DSW(X, C2, T0, S0):
    return np.max(np.array([0,C2*(EOS(X[1],X[3])-EOS(T0,S0))]))

# compute water column stability [kg/m^3]
def compute_Sigma(X):
    return EOS(X[0],X[2])-EOS(X[1],X[3])

# Compute melt rate averaged over all the cavity boxes [m/30d]
def compute_m_avg(m,frac,nbox):
    return np.average(m, weights=frac[:nbox])

def BoxModel(X, nbox, C, gammaT, Ac, Omega, T0, S0, Ap, kappa, g, frac, depth, C2, T_surf, S_surf, AABW):
    
    #compute k 
    k = kappa/gammaT

    #compute phi
    phi = Ap/Ac

    #define variables
    Tp, Td, Sp, Sd = X

    # Cavity dynamics with PICO 
    Tc,Sc,m,q = PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)

    chi = q/Ac/gammaT

    # Dense Shelf Water flux
    if AABW:
        if k>2: # Here we consider we are in the convective mode so we create AABW
            DSW = compute_DSW(X, C2, T0, S0)/Ac/gammaT
        else:
            DSW = 0
        
    # create an empty vector that will contains the dynamical system
    vect_out = np.zeros(len(X))

    if AABW:
        # polynya box equations
        vect_out[0] = chi*(Tc[-1]-Tp)-phi*k*(Tp-Td)+phi*g*(liquidus(Sp,0)-Tp) +DSW*(T_surf-Tp)
        vect_out[2] = chi*(Sc[-1]-Sp)-phi*k*(Sp-Sd)+phi*rho_i/rho_star*Sp/gammaT*Omega +DSW*(S_surf-Sp)
    
        # deep box equations
        vect_out[1] = chi*(T0-Td)+phi*k*(Tp-Td) -DSW*(Td-Tp)
        vect_out[3] = chi*(S0-Sd)+phi*k*(Sp-Sd) -DSW*(Sd-Sp)

    else: #without AABW formation
        # polynya box equations
        vect_out[0] = chi*(Tc[-1]-Tp)-phi*k*(Tp-Td)+phi*g*(liquidus(Sp,0)-Tp)
        vect_out[2] = chi*(Sc[-1]-Sp)-phi*k*(Sp-Sd)+phi*rho_i/rho_star*Sp/gammaT*Omega
    
        # deep box equations
        vect_out[1] = chi*(T0-Td)+phi*k*(Tp-Td)
        vect_out[3] = chi*(S0-Sd)+phi*k*(Sp-Sd)

    return vect_out

In [19]:
Dataset= xr.Dataset(
    {
        "name_isf": (["Nisf"],name_isf,{"descr":"Name of the ice shelves"}),
        "Tp": (["Nisf","regime"],np.zeros((9,3)),{"descr":"°C"}),
        "Sp": (["Nisf","regime"],np.zeros((9,3)),{"descr":"PSU"}),
        "Td": (["Nisf","regime"],np.zeros((9,3)),{"descr":"°C"}),
        "Sd": (["Nisf","regime"],np.zeros((9,3)),{"descr":"PSU"}),
        "Tc": (["Nisf","regime","box"],np.zeros((9,3,5)),{"descr":"°C"}),
        "Sc": (["Nisf","regime","box"],np.zeros((9,3,5)),{"descr":"PSU"}),
        "melt": (["Nisf","regime"],np.zeros((9,3)),{"descr":"m/30d"}),
        "overt": (["Nisf","regime"],np.zeros((9,3)),{"descr":"Sv"}),
        "qAABW": (["Nisf"],np.zeros(9),{"descr":"Sv"}),
        
    },
    coords={
        "Nisf": index_isf,
        "box": np.arange(1,5+1),
        "regime": np.array(['diff','conv','conv_AABW'])
    },
    attrs={
        "Global": "Dataset containing the output parameters for the box model",
        "Nisf": "Index of the ice shelves (from Burgard 2022)",
        "box": "box number (but not ntotal box number)",
    }
)

In [20]:
kappa = kappa_diff

AABW = False

for index in index_isf:
    print(' ')
    print(inputs.name_isf.sel(Nisf=index).values)

    nbox = int(inputs.Nbox.sel(Nisf=index).values)
    Ac = inputs.Ac.sel(Nisf=index).values #m^2

    Omega_ref = np.mean(inputs.Omega.sel(Nisf=index).values)/30/24/3600 #m/s
    Ap_ref = np.mean(inputs.Ap.sel(Nisf=index).values) #m
    T0_ref =  np.mean(inputs.T0.sel(Nisf=index).values) #°C
    
    frac = inputs.frac.sel(Nisf=index).values
    depth = inputs.depth.sel(Nisf=index).values
    T_surf_ref = np.mean(inputs.T_surf.sel(Nisf=index).values)
    S_surf_ref = np.mean(inputs.S_surf.sel(Nisf=index).values)
    S0_ref = np.mean(inputs.S0.sel(Nisf=index).values)

    
    guess= np.array([0,0,34.5,34.5])

    X, info, ier, msg = fsolve(BoxModel, guess, args = (nbox,C,gammaT,Ac,Omega_ref,T0_ref,S0_ref,Ap_ref,kappa,g,frac,depth, C2, T_surf_ref, S_surf_ref, AABW), full_output=True,)
    print(msg)

    sigma = compute_Sigma(X)
    print(sigma<0)


    Tp, Td, Sp, Sd = X

    if sigma<0:
        Tc ,Sc ,m,q = PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)
        m_avg = compute_m_avg(m,frac,nbox)

        Tc = np.concatenate((Tc,np.zeros(5-len(Tc))))
        Sc = np.concatenate((Sc,np.zeros(5-len(Tc))))

        Dataset['Tp'].loc[index,"diff"] = Tp
        Dataset['Td'].loc[index,"diff"] = Td
        Dataset['Sp'].loc[index,"diff"] = Sp
        Dataset['Sd'].loc[index,"diff"] = Sd
        Dataset['Tc'].loc[index,"diff"] = Tc
        Dataset['melt'].loc[index,"diff"] = m_avg
        Dataset['overt'].loc[index,"diff"] = q/1e6
        
        
        

 
Ross
The solution converged.
True
 
Filchner-Ronne
The solution converged.
True
 
Amery
The solution converged.
True
 
Dotson
The solution converged.
False
 
Pine Island
The solution converged.
True
 
Riiser-Larsen
The solution converged.
True
 
Roi Baudouin
The solution converged.
False
 
Totten
The solution converged.
False
 
Moscow Univ.
The solution converged.
True


/tmp/ipykernel_1127/1731552375.py:8: RuntimeWarning: invalid value encountered in sqrt
  x0 = -g1[0]/(2*C*rho_star*(beta*s-alpha))+np.sqrt((g1[0]/(2*C*rho_star*(beta*s-alpha)))**2-g1[0]*T_st_0/(C*rho_star*(beta*s-alpha)))


In [21]:
kappa = kappa_conv

AABW = False

for index in index_isf:
    print(' ')
    print(inputs.name_isf.sel(Nisf=index).values)

    nbox = int(inputs.Nbox.sel(Nisf=index).values)
    Ac = inputs.Ac.sel(Nisf=index).values #m^2

    Omega_ref = np.mean(inputs.Omega.sel(Nisf=index).values)/30/24/3600 #m/s
    Ap_ref = np.mean(inputs.Ap.sel(Nisf=index).values) #m
    T0_ref =  np.mean(inputs.T0.sel(Nisf=index).values) #°C
    
    frac = inputs.frac.sel(Nisf=index).values
    depth = inputs.depth.sel(Nisf=index).values
    T_surf_ref = np.mean(inputs.T_surf.sel(Nisf=index).values)
    S_surf_ref = np.mean(inputs.S_surf.sel(Nisf=index).values)
    S0_ref = np.mean(inputs.S0.sel(Nisf=index).values)

    
    guess= np.array([0,0,34.5,34.5])

    X, info, ier, msg = fsolve(BoxModel, guess, args = (nbox,C,gammaT,Ac,Omega_ref,T0_ref,S0_ref,Ap_ref,kappa,g,frac,depth, C2, T_surf_ref, S_surf_ref, AABW), full_output=True)
    print(msg)

    sigma = compute_Sigma(X)
    print(sigma>0)


    Tp, Td, Sp, Sd = X

    if sigma>0:
        Tc ,Sc ,m,q = PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)
        m_avg = compute_m_avg(m,frac,nbox)

        Tc = np.concatenate((Tc,np.zeros(5-len(Tc))))
        Sc = np.concatenate((Sc,np.zeros(5-len(Tc))))

        Dataset['Tp'].loc[index,"conv"] = Tp
        Dataset['Td'].loc[index,"conv"] = Td
        Dataset['Sp'].loc[index,"conv"] = Sp
        Dataset['Sd'].loc[index,"conv"] = Sd
        Dataset['Tc'].loc[index,"conv"] = Tc
        Dataset['melt'].loc[index,"conv"] = m_avg
        Dataset['overt'].loc[index,"conv"] = q/1e6

 
Ross
The solution converged.
True
 
Filchner-Ronne
The solution converged.
True
 
Amery
The solution converged.
True
 
Dotson
The solution converged.
True
 
Pine Island
The solution converged.
True
 
Riiser-Larsen
The solution converged.
True
 
Roi Baudouin
The solution converged.
True
 
Totten
The solution converged.
True
 
Moscow Univ.
The solution converged.
False


In [23]:
kappa = kappa_conv

AABW = True

for index in index_isf:
    print(' ')
    print(inputs.name_isf.sel(Nisf=index).values)

    nbox = int(inputs.Nbox.sel(Nisf=index).values)
    Ac = inputs.Ac.sel(Nisf=index).values #m^2

    Omega_ref = np.mean(inputs.Omega.sel(Nisf=index).values)/30/24/3600 #m/s
    Ap_ref = np.mean(inputs.Ap.sel(Nisf=index).values) #m
    T0_ref =  np.mean(inputs.T0.sel(Nisf=index).values) #°C
    
    frac = inputs.frac.sel(Nisf=index).values
    depth = inputs.depth.sel(Nisf=index).values
    T_surf_ref = np.mean(inputs.T_surf.sel(Nisf=index).values)
    S_surf_ref = np.mean(inputs.S_surf.sel(Nisf=index).values)
    S0_ref = np.mean(inputs.S0.sel(Nisf=index).values)

    
    guess= np.array([0,0,34.5,34.5])

    X, info, ier, msg = fsolve(BoxModel, guess, args = (nbox,C,gammaT,Ac,Omega_ref,T0_ref,S0_ref,Ap_ref,kappa,g,frac,depth, C2, T_surf_ref, S_surf_ref, AABW), full_output=True)
    print(msg)

    sigma = compute_Sigma(X)
    print(sigma>0)


    Tp, Td, Sp, Sd = X

    if sigma>0:
        Tc ,Sc ,m,q = PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)
        m_avg = compute_m_avg(m,frac,nbox)

        Tc = np.concatenate((Tc,np.zeros(5-len(Tc))))
        Sc = np.concatenate((Sc,np.zeros(5-len(Tc))))

        qAABW = compute_DSW(X, C2, T0_ref, S0_ref)

        Dataset['Tp'].loc[index,"conv_AABW"] = Tp
        Dataset['Td'].loc[index,"conv_AABW"] = Td
        Dataset['Sp'].loc[index,"conv_AABW"] = Sp
        Dataset['Sd'].loc[index,"conv_AABW"] = Sd
        Dataset['Tc'].loc[index,"conv_AABW"] = Tc
        Dataset['melt'].loc[index,"conv_AABW"] = m_avg
        Dataset['overt'].loc[index,"conv_AABW"] = q/1e6
        Dataset['qAABW'].loc[index] = qAABW/1e6

 
Ross
The solution converged.
True
0.4620218064860638
34.91683845107597
 
Filchner-Ronne
The solution converged.
True
0.11179977254846563
34.51647171360628
 
Amery
The solution converged.
True
0.20476266406421928
34.666043940096955
 
Dotson
The solution converged.
True
0.15166103631167968
34.5569542938394
 
Pine Island
The solution converged.
True
0.038190771342669905
34.38486007152343
 
Riiser-Larsen
The solution converged.
True
0.046270131899518674
34.39824698377249
 
Roi Baudouin
The solution converged.
True
0.1148058144353854
34.45471213183788
 
Totten
The solution converged.
True
0.08975346935085327
34.506615816916934
 
Moscow Univ.
The solution converged.
False


In [1]:
Dataset.to_netcdf("clean_output_dataset.nc")

In [78]:
for index in inputs.Nisf.values:
    data_d = Dataset.sel(Nisf=index, regime='diff')
    m_d = str(np.round(data_d.melt.values*12,2))
    q_d = str(np.round(data_d.overt.values,2))
    data_c = Dataset.sel(Nisf=index, regime='conv')
    data_AABW = Dataset.sel(Nisf=index, regime='conv_AABW')
    m_c = str(np.round(data_c.melt.values*12,2))
    q_c = str(np.round(data_c.overt.values,2))
    qAABW = str(np.round(data_AABW.qAABW.values,2))
    print(data_d.name_isf.values+' & '+m_d+ ' & '+q_d+' & '+m_c+ ' & '+q_c+' & '+qAABW+' \\\\')

Ross & 2.7 & 0.83 & 0.01 & 0.13 & 0.71 \\
Filchner-Ronne & 1.17 & 0.6 & 0.04 & 0.22 & 0.16 \\
Amery & 8.06 & 0.51 & 0.44 & 0.17 & 0.33 \\
Dotson & 0.0 & 0.0 & 0.95 & 0.05 & 0.18 \\
Pine Island & 21.25 & 0.21 & 2.15 & 0.08 & 0.04 \\
Riiser-Larsen & 2.25 & 0.19 & 0.17 & 0.07 & 0.06 \\
Roi Baudouin & 0.0 & 0.0 & 0.11 & 0.05 & 0.15 \\
Totten & 0.0 & 0.0 & 2.13 & 0.1 & 0.12 \\
Moscow Univ. & 5.66 & 0.1 & 0.0 & 0.0 & 0.0 \\
